In [155]:
########################################################
#  Distance estimation for qudit twisted-torus codes over GF(q)
#  ---------------------------------------------------------------
#
#  What this script does
#  ---------------------
#  For every code found by the Sage search (one row of the file
#  minus_n{n}fg_{k}k_{q}q_{w}wait.csv), it builds the check matrices
#
#      H_X = [ f(X,Y) | g(X,Y) ],    H_Z = [ gbar(X,Y) | -fbar(X,Y) ]
#
#  on the twisted torus with lattice vectors a1 = (0,alpha), a2 = (beta,gamma),
#  and estimates the code distance with DistRandCSS from the GAP package
#  QDistRnd.  The distance is estimated twice, once for Z-type and once for
#  X-type logical operators, and the smaller value is kept.  For every code
#  the untwisted torus (gamma = 0) with the same f, g, alpha, beta is
#  evaluated as well, so that the effect of the twist can be seen directly.
#
#  DistRandCSS returns the smallest weight of a logical operator found among
#  `num` random information sets; the result is therefore an UPPER BOUND on
#  the true distance, and a larger `num` can only lower it.  
#
#  Input  (written by the Sage notebook):
#      data/data_{n}n_{k}k_{q}q_{w}w/minus_n{n}fg_{k}k_{q}q_{w}wait.csv
#      columns: f, g, n, alpha, beta, gamma, k_gamma0, k_gamma, f_clean,
#               g_clean, f_shift_x, f_shift_y, g_shift_x, g_shift_y,
#               f_no_neg, g_no_neg
#      (f_no_neg, g_no_neg are f, g multiplied by a monomial so that all
#       exponents are >= 0; this only translates the check pattern on the
#       lattice and defines the same code.)
#
#  Output (read by the post-processing notebook):
#      data/data_{n}n_{k}k_{q}q_{w}w/dis_n{n}_{k}k_{q}q_{w}wait.csv
#      columns: f, g, f_no_neg, g_no_neg, alpha, beta, gamma, n,
#               k_gamma0, k_gamma, d0, d, ratio0, ratio
#      where d0 is the distance on the untwisted torus, d on the twisted
#      one, and ratio0 = k_gamma0*d0^2/n, ratio = k_gamma*d^2/n.
#
#  Reference
#  ---------
#  This script accompanies
#      M. Halla, "Qudit Twisted-Torus Codes in the Bivariate Bicycle
#      Framework", arXiv:2602.04443, https://arxiv.org/abs/2602.04443
#  Please cite this paper if you use the script or the codes found with it.
#  Distances: L. P. Pryadko, V. A. Shabashov, V. K. Kozin, "QDistRnd: A GAP
#  package for computing the distance of quantum error-correcting codes",
#  J. Open Source Softw. 7(71), 4120 (2022).
#
#  Software: GAP 4.15.1 with QDistRnd 0.9.5.
#  Copyright (c) 2026 Mourad Halla.  Licence: MIT (see LICENSE).
########################################################

########################################################
# Parameters
########################################################
q:=3;;          # local dimension of the qudits (prime); all matrices are over GF(q)

w:=6;;          # check weight of the ansatz; only used in the file names

k_tg:=4;;       # target number of logical qudits; only used in the file names

F := GF(q);;   # the field GF(q)

########################################################
# Small matrix helpers
########################################################

# Kronecker product of two matrices (kept for completeness; not used below)
Kron := function(A,B)
  local rA,cA,rB,cB,i,j,bi,k,block,rows;
  rA := Length(A);  cA := Length(A[1]);
  rB := Length(B);  cB := Length(B[1]);
  rows := [];
  for i in [1..rA] do
    for bi in [1..rB] do
      block := [];
      for j in [1..cA] do
        block := Concatenation(block,
                    List([1..cB], k -> A[i][j]*B[bi][k]));
      od;
      Add(rows, block);
    od;
  od;
  return rows;
end;;

# r x c zero matrix over F
ZeroMatF := function(r,c)
  return List([1..r], _ -> List([1..c], _ -> Zero(F)));
end;;

# n x n identity matrix over F
IdF := n -> IdentityMat(n,F);;

# horizontal concatenation [A | B]
HCat := function(A,B)
  return List([1..Length(A)], i -> Concatenation(A[i],B[i]));
end;;

########################################################
# Shift matrices X, Y of the twisted torus with basis (0,α), (β,γ)
#
# Unit cells are numbered i = x + β*y + 1 for 0<=x<β, 0<=y<α.
# Y moves a cell one step in the y direction (modulo α).
# X moves a cell one step in the x direction; leaving the last column
# (x = β-1) brings it back to column 0 with the row shifted by -γ, which
# implements the identification x^β y^γ = 1.  With γ = 0 this is the
# ordinary (untwisted) α x β torus.
########################################################

XY := function(alpha,beta,gamma)
  local n,X,Y,x,y,i,x2,y2,j,gmod;
  n := alpha*beta;
  gmod := ((gamma mod alpha) + alpha) mod alpha;   # 0..α-1
  X := ZeroMatF(n,n);
  Y := ZeroMatF(n,n);

  for y in [0..alpha-1] do
    for x in [0..beta-1] do
      i := x + beta*y + 1;

      # y-shift: (x,y) -> (x, y+1)
      x2 := x;          y2 := (y + 1) mod alpha;
      j  := x2 + beta*y2 + 1;
      Y[i][j] := One(F);

      # x-shift with twist only at boundary: X^β = Y^(-γ)
      if x < beta-1 then
        x2 := x + 1;    y2 := y;
      else
        x2 := 0;        y2 := (y - gmod) mod alpha;
      fi;
      j := x2 + beta*y2 + 1;
      X[i][j] := One(F);
    od;
  od;

  return [X,Y];
end;;

# M^k for any integer k; negative k uses the inverse (X, Y are permutation
# matrices, so the inverse always exists)
MatrixPowSigned := function(M,k)
  if k = 0 then
    return IdF(Length(M));
  elif k > 0 then
    return M^k;
  else
    return (M^-1)^(-k);  # negative exponent
  fi;
end;;

########################################################
# Evaluating a polynomial on the torus
########################################################

# the matrix of the monomial x^ax y^ay, i.e. X^ax * Y^ay
MonomialMatrixXY := function(ax, ay, X, Y)
  return MatrixPowSigned(X, ax) * MatrixPowSigned(Y, ay);
end;;

# the matrix p(X,Y) of a polynomial given as a list of terms [ax, ay, coeff]
PolyMatrix := function(terms, alpha, beta, pair)
  local M, t, X, Y;
  X := pair[1];;
  Y := pair[2];;
  M := ZeroMatF(alpha*beta, alpha*beta);
  for t in terms do
    M := M + t[3] * MonomialMatrixXY(t[1], t[2], X, Y);
  od;
  return M;
end;;

########################################################
# Check matrices  H_X = [ f | g ],  H_Z = [ gbar | -fbar ]
#
# gbar, fbar are the antipodes (every exponent negated).  The relative
# minus sign in H_Z is required for odd q: without it the X and Z checks
# do not commute (Lemma 1 of the paper).  DistRandCSS verifies that
# H_X * H_Z^T = 0 before computing anything.
########################################################

BuildHxHz := function(f,g,alpha,beta,gamma)
  local pair, Mf, Mg, fbar, gbar, Mf_bar, Mg_bar, Hx, Hz;

  # X,Y shifts on twisted torus
  pair := XY(alpha,beta,gamma);;
  Mf := PolyMatrix(f, alpha, beta, pair);;
  Mg := PolyMatrix(g, alpha, beta, pair);;

  # antipode: (a,b) -> (-a,-b)  (bar(f), bar(g))
  fbar := List(f, t -> [ -t[1], -t[2], t[3] ]);;
  gbar := List(g, t -> [ -t[1], -t[2], t[3] ]);;

  Mf_bar := PolyMatrix(fbar, alpha, beta, pair);;
  Mg_bar := PolyMatrix(gbar, alpha, beta, pair);;

  # Hx = [ f  g ],  Hz = [ ḡ  -f̄ ]
  Hx := HCat(Mf, Mg);;
    Hz := HCat(Mg_bar,-Mf_bar);;

  return [Hx, Hz];
end;;

########################################################
# Polynomial -> list of terms [ax, ay, coeff]
# Works for polynomials in x, y with nonnegative exponents, which is why
# the script reads the columns f_no_neg, g_no_neg.
########################################################

PolyToXYTermList := function(p, x, y)
  local tmp, terms, term, c, mon, ax, ay, ord;

  ord   := MonomialLexOrdering();
  tmp   := p;
  terms := [];

  while not IsZero(tmp) do
    # leading term = c * monomial
    term := LeadingTermOfPolynomial(tmp, ord);
    c    := LeadingCoefficient(term);

    # monic monomial part to read exponents
    mon  := term / c;
    ax   := DegreeIndeterminate(mon, x);
    ay   := DegreeIndeterminate(mon, y);

    # store [ax, ay, coefficient]
    Add(terms, [ax, ay, c]);

    tmp := tmp - term;
  od;

  return terms;
end;;

########################################################
# Polynomial ring F[x,y] and string parsing
########################################################

R := PolynomialRing(F, ["x","y"]);;
indets := IndeterminatesOfPolynomialRing(R);;
x := indets[1];;
y := indets[2];;

# Turn a polynomial string from the CSV file, e.g. "x^5*y+x^3*y+1",
# into an element of R (the variables x, y above are used by EvalString).
StringToPolynomial := function(str)
  local s, p;
  s := ReplacedString(str, " ", "");
  s := ReplacedString(s, "\t", "");
  s := ReplacedString(s, "\\", "");  # remove line-wrap backslashes

  p := EvalString(s);

  # if EvalString gives a constant in GF(q), embed it into R
  if not IsPolynomial(p) then
    p := p * One(R);
  fi;

  return p;
end;;

########################################################
# Distance estimation with QDistRnd
########################################################

LoadPackage("QDistRnd");;;

num    := 30000;;   # number of information sets; 
mindist:= 0;;     # 0 = search for the distance without an early-stop bound
debug  := 0;;     # verbosity of DistRandCSS: 0 = no output; larger values
                  # print progress and extra checks (see the QDistRnd manual)

# make f,g global so GAP doesn't warn
f := [];;
g := [];;

# Distance of the code (f, g) on the torus (alpha, beta, gamma):
# the smaller of the Z-distance (kernel of H_X modulo rows of H_Z) and the
# X-distance (kernel of H_Z modulo rows of H_X), each an upper bound found
# with `num` information sets.  Uses the GLOBAL term lists f, g.
DistanceForTriple := function(alpha,beta,gamma)
  local pair,Hx_local,Hz_local,dX_local,dZ_local;
  pair := BuildHxHz(f,g,alpha,beta,gamma);;
  Hx_local := pair[1];;
  Hz_local := pair[2];;

  dZ_local := DistRandCSS(Hx_local, Hz_local,
                          num, mindist, debug : field := F);;
  if dZ_local = fail then
    return fail;
  fi;

  dX_local := DistRandCSS(Hz_local, Hx_local,
                          num, mindist, debug : field := F);;
  if dX_local = fail then
    return fail;
  fi;

  return Minimum(dX_local, dZ_local);
end;;

########################################################
# Format a float with exactly two decimals (for the ratio columns)
########################################################

Format2 := function(r)
  local tmp, intp, frac;
  tmp  := Int(100.0 * r + 0.5);   # round
  intp := QuoInt(tmp, 100);
  frac := tmp mod 100;
  if frac < 10 then
    return Concatenation(String(intp), ".0", String(frac));
  else
    return Concatenation(String(intp), ".", String(frac));
  fi;
end;;

########################################################
# One complete run for a given n_target:
# read minus_n...wait.csv, estimate d0 (untwisted) and d (twisted) for
# every row, write dis_n...wait.csv
########################################################

RunDistance := function(n_target)
  local dir, inName, file, line, results, outName, out, parts,
        f_str, g_str, alpha_c, beta_c, gamma_c, k0_c, k_c,
        f_no_neg_str, g_no_neg_str, n_c, f_poly, g_poly,
        d0_c, d_c, ratio0_c, d0_str, ratio0_str, ratio_c, d_str, ratio_str;

  dir := Concatenation(
    "data/data_", String(n_target), "n_",
    String(k_tg), "k_",
    String(q), "q_",
    String(w), "w/"
  );;

  inName := Concatenation(
      "minus_n",
      String(n_target),
      "fg_",
      String(k_tg),
      "k_",
      String(q),
      "q_",
      String(w),
      "wait.csv"
  );;

  file := InputTextFile( Concatenation(dir, inName) );

  if file = fail then
      Print("Cannot open file: ", inName, "  -- skipping n_target = ", n_target, "\n");
      return;
  fi;

  # skip header
  line := ReadLine(file);;

  results := [];;

  outName := Concatenation(
    "dis_n", String(n_target),
    "_", String(k_tg), "k_",
    String(q), "q_",
    String(w),
    "wait.csv"
  );;

  out := OutputTextFile( Concatenation(dir, outName), false );

  if out = fail then
      Error(Concatenation("Cannot open output file ", outName, "\n"));
  fi;

  WriteLine(out, "f,g,f_no_neg,g_no_neg,alpha,beta,gamma,n,k_gamma0,k_gamma,d0,d,ratio0,ratio");
  Print("f,g,alpha,beta,gamma,n,k_gamma0,k_gamma,d0,d,ratio0,ratio\n");

  while not IsEndOfStream(file) do
    line := ReadLine(file);
    if line = fail then
      break;
    fi;

    if Length(line) = 0 then
      continue;
    fi;

    # strip CR/LF
    line := ReplacedString(line, "\r", "");
    line := ReplacedString(line, "\n", "");
    if line = "" then
      continue;
    fi;

    parts := SplitString(line, ",");
    # header (from the Sage "eliminate minus" step):
    # f,g,n,alpha,beta,gamma,k_gamma0,k_gamma,
    # f_clean,g_clean,f_shift_x,f_shift_y,
    # g_shift_x,g_shift_y,f_no_neg,g_no_neg
    if Length(parts) < 16 then
      continue;
    fi;

    # original with (possibly) negative exponents
    f_str := parts[1];
    g_str := parts[2];

    # geometry parameters
    alpha_c := Int(parts[4]);;
    beta_c  := Int(parts[5]);;
    gamma_c := Int(parts[6]);;
    k0_c    := Int(parts[7]);;   # k_gamma0
    k_c     := Int(parts[8]);;   # k_gamma

    if alpha_c = fail or beta_c = fail or
       gamma_c = fail or k0_c = fail or k_c = fail then
      continue;
    fi;

    # non-negative versions
    f_no_neg_str := parts[15];
    g_no_neg_str := parts[16];

    # n from geometry (matches other code)
    n_c := 2 * alpha_c * beta_c;

    # build f,g term lists from f_no_neg,g_no_neg  (f, g are GLOBAL)
    f_poly := StringToPolynomial(f_no_neg_str);;
    g_poly := StringToPolynomial(g_no_neg_str);;

    f := PolyToXYTermList(f_poly, x, y);;
    g := PolyToXYTermList(g_poly, x, y);;

    # distance for gamma=0 (untwisted) and for given gamma
    d0_c := DistanceForTriple(alpha_c, beta_c, 0);;
    d_c  := DistanceForTriple(alpha_c, beta_c, gamma_c);;

    # ratios kd^2/n (with k_gamma0 and k_gamma)
    if d0_c = fail or d0_c > n_c then
      ratio0_c   := fail;
      d0_str     := "fail";
      ratio0_str := "fail";
    else
      ratio0_c   := 1.0 * k0_c * d0_c * d0_c / n_c;
      d0_str     := String(d0_c);
      ratio0_str := Format2(ratio0_c);
    fi;

    if d_c = fail or d_c > n_c then
      ratio_c   := fail;
      d_str     := "fail";
      ratio_str := "fail";
    else
      ratio_c   := 1.0 * k_c * d_c * d_c / n_c;
      d_str     := String(d_c);
      ratio_str := Format2(ratio_c);
    fi;

    # print to screen
    Print(f_str, ",", g_str, ",",
          alpha_c, ",", beta_c, ",", gamma_c, ",",
          n_c, ",",
          k0_c, ",", k_c, ",",
          d0_str, ",", d_str, ",",
          ratio0_str, ",", ratio_str, "\n");

    # write to output CSV
    WriteLine(out,
      Concatenation(
        f_str, ",",
        g_str, ",",
        f_no_neg_str, ",",
        g_no_neg_str, ",",
        String(alpha_c), ",",
        String(beta_c), ",",
        String(gamma_c), ",",
        String(n_c), ",",
        String(k0_c), ",",
        String(k_c), ",",
        d0_str, ",",
        d_str, ",",
        ratio0_str, ",",
        ratio_str));

    Add(results, rec(
      f        := f_str,
      g        := g_str,
      f_no_neg := f_no_neg_str,
      g_no_neg := g_no_neg_str,
      alpha    := alpha_c,
      beta     := beta_c,
      gamma    := gamma_c,
      n        := n_c,
      k_gamma0 := k0_c,
      k_gamma  := k_c,
      d0       := d0_c,
      d        := d_c,
      ratio0   := ratio0_c,
      ratio    := ratio_c));
  od;;

  CloseStream(file);;
  CloseStream(out);;
  Print("done n_target = ", n_target, "\n");
end;;

########################################################
# Run one n after the other
########################################################

n_list := [42];;     # <-- list of n qudit

for n_target in n_list do
  RunDistance(n_target);
od;

f,g,alpha,beta,gamma,n,k_gamma0,k_gamma,d0,d,ratio0,ratio
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,-6,42,4,4,5,6,2.38,3.43
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,-4,42,4,4,5,5,2.38,2.38
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,-2,42,4,4,5,7,2.38,4.67
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,-1,42,4,4,5,5,2.38,2.38
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,0,42,4,4,5,5,2.38,2.38
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,1,42,4,4,5,6,2.38,3.43
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,3,42,4,4,5,5,2.38,2.38
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,5,42,4,4,5,7,2.38,4.67
1 + x^-1 + x^-2*y^-2,x^3*y^-3 + y^2 + 1,7,3,6,42,4,4,5,5,2.38,2.38
1 + x^-1*y^-3 + x^-3,x^3*y^-2 + x^2*y^2 + 1,3,7,0,42,4,4,6,6,3.43,3.43
x^3*y + 1 + y^-1,x^3*y^3 + 1 + x^-1*y^3,3,7,0,42,4,4,5,5,2.38,2.38
y + 1 + x^-2*y^-3,x^2*y^-3 + 1 + x^-2,3,7,-2,42,2,4,6,6,1.71,3.43
y + 1 + x^-2*y^-3,x^2*y^-3 + 1 + x^-2,3,7,1,42,2,4,6,6,1.71,3.43
1 + x^-1 + x^-2,x^2*y^-1 + 1 + y^-2,3,7,0,42,4,4,7,7,4.67,4.67
1 +

1 + x^-1*y^2 + x^-2*y,1 + x^-2*y^2 + x^-2*y^-3,21,1,-5,42,2,4,7,6,2.33,3.43
1 + x^-1*y^2 + x^-2*y,1 + x^-2*y^2 + x^-2*y^-3,21,1,4,42,2,4,7,6,2.33,3.43
1 + x^-1*y^2 + x^-2*y,1 + x^-2*y^2 + x^-2*y^-3,21,1,7,42,2,4,7,5,2.33,2.38
1 + x^-1*y^2 + x^-2*y,1 + x^-2*y^2 + x^-2*y^-3,21,1,13,42,2,4,7,7,2.33,4.67
1 + x^-1*y^2 + x^-2*y,1 + x^-2*y^2 + x^-2*y^-3,21,1,16,42,2,4,7,6,2.33,3.43
x^3*y^2 + x^3*y + 1,1 + y^-1 + x^-2*y^-1,3,7,-2,42,2,4,8,6,3.05,3.43
x^3*y^2 + x^3*y + 1,1 + y^-1 + x^-2*y^-1,3,7,1,42,2,4,8,6,3.05,3.43
x^3*y^2 + x^2*y^2 + 1,x^2*y^-3 + x*y^-3 + 1,3,7,-1,42,2,4,7,4,2.33,1.52
x^3*y^2 + x^2*y^2 + 1,x^2*y^-3 + x*y^-3 + 1,3,7,2,42,2,4,7,4,2.33,1.52
x^2*y^-1 + x*y^-2 + 1,1 + x^-1*y^3 + x^-1*y,3,7,-2,42,2,4,5,5,1.19,2.38
x^2*y^-1 + x*y^-2 + 1,1 + x^-1*y^3 + x^-1*y,3,7,1,42,2,4,5,5,1.19,2.38
x^2*y^-1 + x*y^-2 + 1,1 + x^-1*y^3 + x^-1*y,21,1,-17,42,2,4,5,7,1.19,4.67
x^2*y^-1 + x*y^-2 + 1,1 + x^-1*y^3 + x^-1*y,21,1,-14,42,2,4,5,7,1.19,4.67
x^2*y^-1 + x*y^-2 + 1,1 + x^-1*y^3 + x^-1*y,21,1,-8